In [ ]:
"""
Panel Event Study Analysis Template
=====================================
- 데이터: 여러 언어/그룹 × 주차별 시계열 (패널 구조)
- Breakpoint: week = 0 (ChatGPT 출시, 2022-11-30)
- 시간 단위: 월(4주) 단위 bin
- Reference period: 이벤트 직전 1개월 (month_bin = -1)
- Model: Two-Way Fixed Effects (TWFE) Event Study

사용법:
  1. 아래 [USER CONFIG] 섹션에서 파일 경로, 컬럼명 등을 수정하세요.
  2. python event_study_template.py 로 실행하세요.

필요 패키지:
  pip install pandas numpy statsmodels matplotlib linearmodels
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

from lib.utils.file_io import *
from setting_for_sdm.path_setting import path_list
from setting_for_sdm.date_setting import Date_Setting

warnings.filterwarnings("ignore")



def load_and_prepare(file_path, week_col, y_col, group_col, breakpoint, bin_size):
    """데이터 로드 및 이벤트 스터디 변수 생성"""
    # --- 파일 로드 ---
    run_id_start = 10000
    lang = 'python'
    print(f'[visualizing....] start visualizing {lang} language')
    viz_dir = f'{path_list["data_root_dir"]}/result/code_complexity/run_id_{run_id_start}'
    option_dict = load_json(f"{viz_dir}/data/option.json")

    std_date = Date_Setting[option_dict['year_range']]['std_date']

    df = pd.read_parquet(f'{option_dict["save_dir"]}/data/all_complexity.parquet')
    df['id'] = df['Path'].apply(lambda x : x.split('_')[1].split('.')[0])
    df[['id', 'Cognitive Complexity']] = df[['id', 'Cognitive Complexity']].astype(int)

    df = df.groupby('id', as_index=False)['Cognitive Complexity'].max()
    df['Cognitive Complexity'] = np.log(df['Cognitive Complexity']+1)

    origin_df = load_df(option_dict['data_dir'], ['id', 'creationdate', 'title','tags', 'body'])


    viz_df = pd.merge(df, origin_df, on = 'id')[['id', 'creationdate', 'Cognitive Complexity']]
    viz_df['rel_week'] = np.floor((pd.to_datetime(viz_df['creationdate'], format='mixed')- std_date).dt.days/7)
    viz_df = (viz_df.groupby('rel_week', as_index=False)['Cognitive Complexity'].std())
    viz_df = viz_df.rename(columns={'Cognitive Complexity': 'complexity'})



    df = df.sort_values([group_col, week_col]).reset_index(drop=True)

    # --- 이벤트 시점 기준 상대 주차 ---
    df["rel_week"] = df[week_col] - breakpoint

    # --- 월(4주) 단위 bin 생성 ---
    # 양수: 0~3주 → bin 0, 4~7주 → bin 1, ...
    # 음수: -1~-4주 → bin -1, -5~-8주 → bin -2, ...
    df["month_bin"] = np.where(
        df["rel_week"] >= 0,
        df["rel_week"] // bin_size,
        -(-df["rel_week"] - 1) // bin_size - 1
    )

    return df


def create_event_dummies(df, reference_bin):
    """이벤트 더미 변수 생성 (reference bin 제외)"""
    bins = sorted(df["month_bin"].unique())
    bins_without_ref = [b for b in bins if b != reference_bin]

    for b in bins_without_ref:
        df[f"bin_{b}"] = (df["month_bin"] == b).astype(int)

    return df, bins_without_ref


def run_event_study_ols(df, y_col, group_col, bins_without_ref):
    """
    OLS with Group & Time Fixed Effects (수동 구현)
    linearmodels 없이도 동작하도록 demeaning 방식 사용
    """
    import statsmodels.api as sm

    # --- 더미 변수명 ---
    dummy_cols = [f"bin_{b}" for b in bins_without_ref]

    # --- Group Fixed Effects: group demeaning ---
    df_dm = df.copy()
    group_means = df_dm.groupby(group_col)[[y_col] + dummy_cols].transform("mean")
    for col in [y_col] + dummy_cols:
        df_dm[f"{col}_dm"] = df_dm[col] - group_means[col]

    # --- Time Fixed Effects: month_bin demeaning ---
    time_means = df_dm.groupby("month_bin")[
        [f"{y_col}_dm"] + [f"{c}_dm" for c in dummy_cols]
    ].transform("mean")
    y_final = df_dm[f"{y_col}_dm"] - time_means[f"{y_col}_dm"]
    X_cols_dm = [f"{c}_dm" for c in dummy_cols]
    X_final = df_dm[X_cols_dm].subtract(
        time_means[[f"{c}_dm" for c in dummy_cols]].values
    )

    # --- OLS with HAC ---
    X_final = sm.add_constant(X_final)
    model = sm.OLS(y_final, X_final).fit(
        cov_type="cluster", cov_kwds={"groups": df[group_col]}
    )

    return model, dummy_cols


def run_event_study_panel(df, y_col, group_col, bins_without_ref):
    """
    linearmodels PanelOLS (설치되어 있을 때 사용)
    Two-Way Fixed Effects: Entity(group) + Time(month_bin)
    """
    try:
        from linearmodels.panel import PanelOLS

        dummy_cols = [f"bin_{b}" for b in bins_without_ref]

        df_panel = df.set_index([group_col, "month_bin"])

        formula = f"{y_col} ~ " + " + ".join(dummy_cols) + " + EntityEffects + TimeEffects"
        model = PanelOLS.from_formula(formula, data=df_panel).fit(
            cov_type="clustered", cluster_entity=True
        )
        return model, dummy_cols, True

    except ImportError:
        print("⚠ linearmodels 미설치 → OLS demeaning 방식으로 대체합니다.")
        print("  (설치: pip install linearmodels)")
        model, dummy_cols = run_event_study_ols(df, y_col, group_col, bins_without_ref)
        return model, dummy_cols, False


def extract_coefficients(model, bins_without_ref, reference_bin, used_panel=False):
    """모델에서 계수와 신뢰구간 추출"""
    results = []

    for b in bins_without_ref:
        col_name = f"bin_{b}"
        if used_panel:
            col_name_lookup = col_name
        else:
            col_name_lookup = f"{col_name}_dm"
            if col_name_lookup not in model.params.index:
                col_name_lookup = col_name

        try:
            coef = model.params[col_name_lookup]
            ci = model.conf_int().loc[col_name_lookup]
            pval = model.pvalues[col_name_lookup]
            results.append({
                "month_bin": b,
                "coef": coef,
                "ci_lower": ci.iloc[0] if hasattr(ci, "iloc") else ci[0],
                "ci_upper": ci.iloc[1] if hasattr(ci, "iloc") else ci[1],
                "pvalue": pval,
            })
        except KeyError:
            continue

    # Reference period 추가 (계수 = 0)
    results.append({
        "month_bin": reference_bin,
        "coef": 0.0,
        "ci_lower": 0.0,
        "ci_upper": 0.0,
        "pvalue": np.nan,
    })

    results_df = pd.DataFrame(results).sort_values("month_bin").reset_index(drop=True)
    return results_df


def plot_event_study(results_df, reference_bin, bin_size, y_col):
    """이벤트 스터디 그래프"""
    fig, ax = plt.subplots(figsize=(14, 7))

    # 신뢰구간 밴드
    ax.fill_between(
        results_df["month_bin"],
        results_df["ci_lower"],
        results_df["ci_upper"],
        alpha=0.2, color="steelblue", label="95% CI"
    )

    # 계수 점 + 선
    ax.plot(
        results_df["month_bin"], results_df["coef"],
        color="steelblue", linewidth=1.5, marker="o", markersize=5,
        label="Coefficient"
    )

    # 유의한 계수 강조
    sig = results_df[results_df["pvalue"] < 0.05]
    ax.scatter(
        sig["month_bin"], sig["coef"],
        color="darkorange", s=60, zorder=5, label="p < 0.05"
    )

    # 기준선
    ax.axhline(y=0, color="black", linewidth=0.8, linestyle="-")
    ax.axvline(x=0, color="red", linewidth=1.2, linestyle="--", alpha=0.7, label="ChatGPT Release")
    ax.axvline(x=reference_bin, color="gray", linewidth=1, linestyle=":", alpha=0.5)

    # Reference period 표시
    ax.annotate(
        f"Reference\n(bin={reference_bin})",
        xy=(reference_bin, 0), xytext=(reference_bin - 3, results_df["ci_upper"].max() * 0.5),
        fontsize=9, ha="center", color="gray",
        arrowprops=dict(arrowstyle="->", color="gray", lw=0.8),
    )

    # 꾸미기
    ax.set_xlabel(f"Months relative to ChatGPT release ({bin_size}-week bins)", fontsize=12)
    ax.set_ylabel(f"Effect on {y_col}\n(relative to reference period)", fontsize=12)
    ax.set_title("Event Study: Dynamic Treatment Effects", fontsize=14, fontweight="bold")
    ax.legend(loc="upper left", fontsize=9)

    # x축 레이블
    all_bins = sorted(results_df["month_bin"].unique())
    tick_step = max(1, len(all_bins) // 20)
    ax.set_xticks(all_bins[::tick_step])

    plt.tight_layout()
    plt.savefig("event_study_result.png", dpi=150, bbox_inches="tight")
    print("\n→ 시각화 저장: event_study_result.png")
    plt.show()


def pre_trend_test(results_df, reference_bin):
    """
    Pre-trend 검정: 이벤트 이전 계수들이 jointly 0인지 F-test
    """
    print("\n" + "=" * 60)
    print("Pre-trend 검정")
    print("=" * 60)

    pre_bins = results_df[
        (results_df["month_bin"] < 0) &
        (results_df["month_bin"] != reference_bin)
    ]

    if len(pre_bins) == 0:
        print("  Pre-period 계수가 없어 검정 불가")
        return

    n_sig = (pre_bins["pvalue"] < 0.05).sum()
    n_total = len(pre_bins)

    print(f"\n  Pre-period bins: {n_total}개")
    print(f"  유의한 계수 (p<0.05): {n_sig}개")

    # 개별 계수 출력
    print(f"\n  {'Bin':>6}  {'Coef':>10}  {'p-value':>10}  {'Sig':>5}")
    print("  " + "-" * 40)
    for _, row in pre_bins.iterrows():
        sig = "***" if row["pvalue"] < 0.001 else "**" if row["pvalue"] < 0.01 else "*" if row["pvalue"] < 0.05 else ""
        print(f"  {row['month_bin']:>6.0f}  {row['coef']:>10.4f}  {row['pvalue']:>10.4f}  {sig:>5}")

    if n_sig / n_total > 0.2:
        print(f"\n  ⚠ 주의: pre-period 계수 중 {n_sig}/{n_total}개가 유의함")
        print("    → parallel trend 가정이 위배될 수 있음")
        print("    → 결과 해석에 주의 필요")
    else:
        print(f"\n  ✓ Pre-period 계수 대부분 비유의 → parallel trend 가정 지지")


def summary_table(results_df):
    """결과 요약 테이블 출력"""
    print("\n" + "=" * 60)
    print("계수 요약 테이블")
    print("=" * 60)

    def sig_label(p):
        if pd.isna(p):
            return "ref"
        if p < 0.001:
            return "***"
        elif p < 0.01:
            return "**"
        elif p < 0.05:
            return "*"
        elif p < 0.1:
            return "†"
        else:
            return ""

    print(f"\n  {'Bin':>6}  {'Coef':>10}  {'CI_lower':>10}  {'CI_upper':>10}  {'p-value':>10}  {'Sig':>5}")
    print("  " + "-" * 60)
    for _, row in results_df.iterrows():
        sig = sig_label(row["pvalue"])
        pval_str = f"{row['pvalue']:.4f}" if not pd.isna(row["pvalue"]) else "   ref"
        print(
            f"  {row['month_bin']:>6.0f}  {row['coef']:>10.4f}  "
            f"{row['ci_lower']:>10.4f}  {row['ci_upper']:>10.4f}  "
            f"{pval_str:>10}  {sig:>5}"
        )


def run_robustness_different_bins(df, y_col, group_col, breakpoint, reference_bin, alt_bin_sizes):
    """
    Robustness check: 다른 bin 크기로 재분석

    alt_bin_sizes: 테스트할 bin 크기 리스트 (예: [2, 4, 8, 13])
    """
    print("\n" + "=" * 60)
    print("Robustness Check: 다른 bin 크기 비교")
    print("=" * 60)

    for bs in alt_bin_sizes:
        print(f"\n--- Bin size = {bs}주 ---")

        df_alt = df.copy()
        df_alt["month_bin"] = np.where(
            df_alt["rel_week"] >= 0,
            df_alt["rel_week"] // bs,
            -(-df_alt["rel_week"] - 1) // bs - 1
        )

        ref_bin = reference_bin  # 동일한 reference bin 사용
        if ref_bin not in df_alt["month_bin"].unique():
            ref_bin = df_alt.loc[df_alt["month_bin"] < 0, "month_bin"].max()
            print(f"  (reference bin 조정: {ref_bin})")

        df_alt, bins_wo_ref = create_event_dummies(df_alt, ref_bin)
        model, dummy_cols, used_panel = run_event_study_panel(
            df_alt, y_col, group_col, bins_wo_ref
        )
        results = extract_coefficients(model, bins_wo_ref, ref_bin, used_panel)

        # Post-period 유의한 계수 비율
        post = results[(results["month_bin"] >= 0) & (~results["pvalue"].isna())]
        n_sig = (post["pvalue"] < 0.05).sum()
        print(f"  Post-period: {n_sig}/{len(post)} bins 유의 (p<0.05)")


# # ============================================================
# # 메인 실행
# # ============================================================
# if __name__ == "__main__":
#     print("=" * 60)
#     print("Panel Event Study Analysis")
#     print("=" * 60)

#     # 1) 데이터 로드 및 준비
#     df = load_and_prepare(FILE_PATH, WEEK_COL, Y_COL, GROUP_COL, BREAKPOINT, BIN_SIZE)

#     groups = df[GROUP_COL].nunique()
#     print(f"\n데이터 로드 완료: {len(df)} rows")
#     print(f"그룹 수: {groups}개 → {df[GROUP_COL].unique().tolist()}")
#     print(f"주차 범위: {df[WEEK_COL].min()} ~ {df[WEEK_COL].max()}")
#     print(f"월 bin 범위: {df['month_bin'].min()} ~ {df['month_bin'].max()}")
#     print(f"Bin 크기: {BIN_SIZE}주")
#     print(f"Reference bin: {REFERENCE_BIN}")

#     # 2) 이벤트 더미 생성
#     df, bins_without_ref = create_event_dummies(df, REFERENCE_BIN)
#     print(f"이벤트 더미 수: {len(bins_without_ref)}개 (reference 제외)")

#     # 3) 패널 이벤트 스터디 회귀분석
#     print("\n" + "=" * 60)
#     print("Two-Way Fixed Effects Event Study Regression")
#     print("=" * 60)
#     model, dummy_cols, used_panel = run_event_study_panel(
#         df, Y_COL, GROUP_COL, bins_without_ref
#     )
#     if used_panel:
#         print(model.summary)
#     else:
#         print(model.summary())

#     # 4) 계수 추출 및 요약
#     results_df = extract_coefficients(model, bins_without_ref, REFERENCE_BIN, used_panel)
#     summary_table(results_df)

#     # 5) Pre-trend 검정
#     pre_trend_test(results_df, REFERENCE_BIN)

#     # 6) 시각화
#     plot_event_study(results_df, REFERENCE_BIN, BIN_SIZE, Y_COL)

#     # 7) 결과를 CSV로 저장
#     results_df.to_csv("event_study_coefficients.csv", index=False)
#     print("→ 계수 테이블 저장: event_study_coefficients.csv")

#     # 8) Robustness check (다른 bin 크기)
#     #    주석 해제하고 실행하세요
#     # run_robustness_different_bins(
#     #     df, Y_COL, GROUP_COL, BREAKPOINT, REFERENCE_BIN,
#     #     alt_bin_sizes=[2, 8, 13]  # 2주, 8주(2개월), 13주(분기)
#     # )